# Notebook 01 — Cadrage, données, analyse exploratoire et audit de fuite
**Projet final · S7MOI1 · Science des Données, concepts avancés · ECAM-EPMI · 2AE**
Équipe : AKONDE Esly · DELBARD-MAZZOLA Paul-Baptiste · DIOP Abdou Aziz

**Données :** *Individual Household Electric Power Consumption*, UCI Machine Learning Repository (id 235) : un foyer, mesures à la minute, de décembre 2006 à novembre 2010.

## Rappel du cadrage
| Élément | Réponse |
|---|---|
| Contexte | Un fournisseur d'électricité achète l'énergie la veille pour un parc de foyers équipés de compteurs communicants. Utilisateur : le responsable des achats d'énergie. Le foyer UCI sert de cas d'étude. |
| Décision | Chaque jour J−1 en fin de journée, fixer le volume d'énergie à acheter pour le jour J (horizon de prévision : 1 jour). |
| Situation actuelle | Sans modèle : « demain = aujourd'hui » ou « même jour la semaine dernière ». Ce sont nos baselines. |
| Critère de succès | Réduire d'au moins 15 % la MAE de la meilleure baseline sur la période de test (la plus récente). |
| Contraintes | Seules les informations connues à la fin de J−1 sont utilisables ; validation dans l'ordre du temps ; modèle explicable ; données d'un seul foyer. |

**Problématique 1 (supervisée, prévision) :** prévoir `conso_kwh` du jour J avec les informations du jour J−1 et antérieures.
**Problématique 2 (non supervisée) :** identifier des profils types de journées (courbe de 24 heures), préparés en section 8 et modélisés dans le notebook 02.

## Plan
0. Mise en place · 1. Chargement · 2. Qualité · 3. Table journalière · 4. Analyse exploratoire · 5. Audit de fuite · 6. Variables construites · 7. Baselines de calibrage · 8. Profils horaires · 9. Bilan

## 0 · Mise en place
Environnement : Google Colab (Python 3). Les données sont téléchargées dans le dossier `data/`, qui est temporaire sur Colab : la cellule de téléchargement les récupère à chaque nouvelle session. Le dépôt ne contient pas les données (fichier de plus de 50 Mo) mais le script de téléchargement. La graine aléatoire est fixée (`SEED = 42`) pour que les résultats soient reproductibles.

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42                      # graine fixe : résultats reproductibles
np.random.seed(SEED)

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

DATA_DIR = "data"
RAW_PATH = os.path.join(DATA_DIR, "household_power_consumption.txt")
URL = ("https://archive.ics.uci.edu/static/public/235/"
       "individual+household+electric+power+consumption.zip")
os.makedirs(DATA_DIR, exist_ok=True)
print("Environnement prêt.")

## 1 · Chargement des données
**Source :** UCI Machine Learning Repository, *Individual Household Electric Power Consumption* (id 235), consulté en octobre 2026.

**Choix de lecture :**
- `sep=";"` : le fichier est séparé par des points-virgules.
- `na_values=["?"]` : les valeurs manquantes sont codées « ? ». Sans cette option, pandas lirait les colonnes comme du texte.
- `Date` et `Time` sont fusionnées en un **horodatage** placé en **index** et **trié** : en série temporelle, l'ordre des lignes porte l'information.

In [ ]:
def telecharger_donnees():
    # Télécharge et décompresse le jeu UCI s'il n'est pas déjà présent.
    if os.path.exists(RAW_PATH):
        print("Fichier déjà présent :", RAW_PATH)
        return
    zip_path = os.path.join(DATA_DIR, "household.zip")
    try:
        print("Téléchargement depuis l'UCI...")
        urllib.request.urlretrieve(URL, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(DATA_DIR)
        print("Téléchargement terminé.")
    except Exception as e:
        print("Échec du téléchargement automatique :", e)
        print("Plan B : téléchargez le zip depuis la page UCI (id 235), puis uploadez ici.")
        try:
            from google.colab import files        # fonctionne seulement dans Colab
            up = files.upload()
            nom = next(iter(up))
            if nom.endswith(".zip"):
                with zipfile.ZipFile(nom) as z:
                    z.extractall(DATA_DIR)
            else:
                os.replace(nom, RAW_PATH)
        except ImportError:
            raise FileNotFoundError("Placez household_power_consumption.txt dans le dossier data/.")

telecharger_donnees()
assert os.path.exists(RAW_PATH), "household_power_consumption.txt introuvable dans data/"

In [ ]:
raw = pd.read_csv(RAW_PATH, sep=";", na_values=["?"], low_memory=False)

# Horodatage = Date + Time ; le jour est écrit sans zéro initial (ex. 1/1/2007)
raw["timestamp"] = pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d/%m/%Y %H:%M:%S")
raw = raw.drop(columns=["Date", "Time"]).set_index("timestamp").sort_index()

print("Forme (lignes, colonnes) :", raw.shape)
print("Début :", raw.index.min(), "| Fin :", raw.index.max())
raw.head()

### Dictionnaire des variables
| Variable | Type | Unité | Signification |
|---|---|---|---|
| `Global_active_power` | float | kW | puissance active moyenne du foyer sur la minute |
| `Global_reactive_power` | float | kW | puissance réactive moyenne sur la minute |
| `Voltage` | float | V | tension moyenne sur la minute |
| `Global_intensity` | float | A | intensité moyenne sur la minute |
| `Sub_metering_1` | float | Wh | énergie de la **cuisine** (lave-vaisselle, four, micro-ondes) |
| `Sub_metering_2` | float | Wh | énergie de la **buanderie** (lave-linge, sèche-linge, réfrigérateur, éclairage) |
| `Sub_metering_3` | float | Wh | énergie du **chauffe-eau électrique et de la climatisation** |

**Passage kW → kWh :** une puissance moyenne de P kW pendant une minute représente P/60 kWh. Sur une journée, **énergie (kWh) = puissance moyenne (kW) × 24 h**.

In [ ]:
print("Types des colonnes :")
print(raw.dtypes)
print()
raw.describe().round(3)

**Interprétation.** La puissance active moyenne est de 1,09 kW pour une médiane de 0,60 kW, un écart-type de 1,06 kW et un maximum de 11,12 kW : le foyer consomme faiblement la plupart du temps, avec des pics ponctuels. La tension est stable (moyenne 240,8 V, écart-type 3,2 V). Les sous-compteurs « cuisine » et « buanderie » ont une médiane de 0 Wh par minute : ces équipements ne fonctionnent que par intermittence.

## 2 · Qualité des données
**Objectif :** détecter doublons, trous et valeurs manquantes, puis décider d'un traitement justifié.

En série temporelle, on distingue deux cas : une **ligne absente** (la minute n'existe pas dans le fichier) et une **ligne présente mais vide** (valeur « ? »). On rend les deux visibles en imposant une fréquence d'une minute (`reindex`).

In [ ]:
# 1. Doublons d'horodatage
print("Horodatages en double :", raw.index.duplicated().sum())

# 2. Minutes absentes du fichier
index_complet = pd.date_range(raw.index.min(), raw.index.max(), freq="min")
print("Minutes attendues :", len(index_complet))
print("Minutes présentes :", len(raw))
print("Minutes absentes  :", len(index_complet) - len(raw))

# On impose la fréquence : une minute absente devient une ligne vide (NaN) visible
raw = raw.reindex(index_complet)
raw.index.name = "timestamp"

# 3. Taux de valeurs manquantes par colonne
taux_manquants = raw.isna().mean().sort_values(ascending=False) * 100
print("\nTaux de valeurs manquantes (%) :")
print(taux_manquants.round(2))

In [ ]:
def blocs_manquants(s):
    # Taille de chaque bloc consécutif de valeurs manquantes
    na = s.isna()
    groupe = na.ne(na.shift()).cumsum()
    tailles = na.groupby(groupe).sum()
    return tailles[tailles > 0]

blocs = blocs_manquants(raw["Global_active_power"])
print("Nombre de blocs manquants :", len(blocs))
print("Plus long bloc (minutes)   :", int(blocs.max()), "soit", round(blocs.max() / 60 / 24, 2), "jours")
print("Blocs d'au moins 1 heure   :", int((blocs >= 60).sum()))
print("Blocs d'au moins 1 jour    :", int((blocs >= 1440).sum()))

# Part de minutes manquantes par mois
manq_mois = raw["Global_active_power"].isna().resample("MS").mean() * 100
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(manq_mois)), manq_mois.values)
ax.set_title("Part de minutes manquantes par mois (Global_active_power)")
ax.set_ylabel("% de minutes manquantes")
ax.set_xlabel("Mois")
ax.set_xticks(range(0, len(manq_mois), 6))
ax.set_xticklabels([d.strftime("%Y-%m") for d in manq_mois.index[::6]], rotation=45)
plt.tight_layout(); plt.show()

**Interprétation.** Le fichier ne contient aucun doublon et aucune minute absente : les 2 075 259 minutes de la période sont toutes présentes, et les trous sont uniquement des valeurs « ? ». Elles représentent 1,25 % des lignes et touchent les sept colonnes en même temps, ce qui indique une interruption de la collecte et non la panne d'un capteur isolé.

Elles ne sont pas dispersées au hasard : elles se regroupent en 71 blocs, dont 9 durent au moins une heure et 6 au moins un jour, le plus long atteignant 7 226 minutes (5,02 jours). Le graphique mensuel montre que la plupart des mois sont quasiment intacts alors que quelques-uns sont fortement touchés (jusqu'à environ 15 % de minutes manquantes sur un mois en 2010).

**Décision de traitement.**
- Nous n'imputons pas à la minute avec la moyenne de toute la série : ce serait une fuite de données (la moyenne contient le futur) et cela reviendrait à inventer des jours entiers.
- Nous agrégeons au jour en calculant la moyenne sur les minutes **valides** et nous déclarons **incomplet** un jour qui a trop de minutes manquantes (section 3).
- Les jours incomplets sont ensuite traités par une règle causale (dernière valeur connue, limitée à 3 jours, section 6).

## 3 · Passage à la table journalière
**Objectif :** transformer 2 millions de lignes à la minute en une ligne par jour. La décision (achat d'énergie à J−1) se prend par jour : prévoir à la minute n'aurait aucun intérêt métier et le bruit serait très élevé.

**Méthode :**
- `conso_kwh` = puissance moyenne du jour (kW) × 24, calculée sur les minutes valides uniquement. Cette formule évite de sous-estimer un jour qui aurait quelques minutes manquantes.
- On conserve la **couverture** (`minutes_valides`) : un jour avec moins de 95 % de minutes valides est marqué `jour_incomplet` et sa consommation est mise à `NaN`, donc visible plutôt que masquée.
- Les trois sous-compteurs sont convertis en kWh par jour, et `reste_kwh` désigne la consommation non mesurée par les sous-compteurs.

In [ ]:
SEUIL_COUVERTURE = 0.95                       # 95 % des 1 440 minutes
SEUIL_MINUTES = int(SEUIL_COUVERTURE * 1440)

gap = raw["Global_active_power"]
daily = pd.DataFrame({
    "minutes_valides": gap.resample("D").count(),
    "puissance_moy_kw": gap.resample("D").mean(),
})
daily["conso_kwh"] = daily["puissance_moy_kw"] * 24
daily["jour_incomplet"] = daily["minutes_valides"] < SEUIL_MINUTES

# Sous-compteurs : Wh par minute -> moyenne x 1440 minutes / 1000 = kWh par jour
for nom, col in [("sub_cuisine_kwh", "Sub_metering_1"),
                 ("sub_buanderie_kwh", "Sub_metering_2"),
                 ("sub_chauffe_kwh", "Sub_metering_3")]:
    daily[nom] = raw[col].resample("D").mean() * 1440 / 1000
daily["reste_kwh"] = daily["conso_kwh"] - daily[["sub_cuisine_kwh", "sub_buanderie_kwh", "sub_chauffe_kwh"]].sum(axis=1)

# Un jour incomplet : toutes ses mesures sont mises à NaN (visibles, pas inventées)
cols_mesures = ["puissance_moy_kw", "conso_kwh", "sub_cuisine_kwh", "sub_buanderie_kwh",
                "sub_chauffe_kwh", "reste_kwh"]
daily.loc[daily["jour_incomplet"], cols_mesures] = np.nan

print("Nombre de jours      :", len(daily))
print("Jours incomplets     :", int(daily["jour_incomplet"].sum()))
print("Premier jour complet :", daily.index[~daily["jour_incomplet"]].min().date())
print("Dernier jour complet :", daily.index[~daily["jour_incomplet"]].max().date())
daily.head(8).round(2)

In [ ]:
# Contrôle : les jours incomplets sont-ils isolés ou groupés ?
blocs_jours = blocs_manquants(daily["conso_kwh"])
print("Blocs de jours manquants :", len(blocs_jours))
print("Tailles (en jours)        :", sorted(blocs_jours.astype(int).tolist(), reverse=True)[:15])
print("Dates des jours incomplets :")
print([d.date().isoformat() for d in daily.index[daily["jour_incomplet"]]][:30])

**Interprétation.** Sur 1 442 jours, 25 (1,7 %) sont déclarés incomplets, répartis en 10 blocs de 1 à 6 jours. Le premier jour (16/12/2006) ne compte que 396 minutes valides parce que l'enregistrement commence à 17:24, et le dernier (26/11/2010) est tronqué parce qu'il s'arrête à 21:02. Il reste **1 417 jours** de consommation fiable, du 17/12/2006 au 25/11/2010.

Nous avons donc agrégé au jour avec une règle de couverture à 95 % et gardé visibles les jours non fiables, plutôt que de combler les trous à la minute.

## 4 · Analyse exploratoire
**Objectif :** observer la cible `conso_kwh` sous quatre angles : son évolution dans le temps, sa distribution, sa saisonnalité (semaine et année) et son autocorrélation. Cela permet de savoir ce que le modèle devra capturer.

*Remarque :* pour **regarder** la série, une moyenne mobile centrée est acceptable (elle lisse le dessin). Pour **prévoir**, elle est interdite car elle contient le futur ; elle est refaite correctement en section 6.

In [ ]:
# Graphique 1 : la série dans le temps
c = daily["conso_kwh"]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(c.index, c.values, linewidth=0.6, label="Consommation journalière")
ax.plot(c.index, c.rolling(30, center=True, min_periods=15).mean(), color="red",
        linewidth=1.5, label="Moyenne mobile 30 jours (lecture seule)")
ax.set_title("Consommation électrique journalière du foyer")
ax.set_xlabel("Date"); ax.set_ylabel("Énergie (kWh par jour)")
ax.legend(); plt.tight_layout(); plt.show()

**Interprétation.** La consommation journalière présente une saisonnalité annuelle marquée : elle est élevée en hiver (la moyenne mobile dépasse 40 kWh début 2007) et chute en été, avec des creux de l'ordre de 5 à 15 kWh en moyenne mobile chaque été, probablement liés à des périodes d'absence du foyer (hypothèse : nos données ne contiennent aucune information sur les vacances). Aucune tendance de fond nette ne se dégage sur les quatre ans. Le début de série (décembre 2006) contient des valeurs très élevées, avec un pic à 79,6 kWh le 23/12/2006.

In [ ]:
# Graphique 2 : distribution de la cible
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(c.dropna(), bins=40, edgecolor="white")
ax.set_title("Distribution de la consommation journalière")
ax.set_xlabel("Énergie (kWh par jour)"); ax.set_ylabel("Nombre de jours")
plt.tight_layout(); plt.show()

print(c.describe().round(2))
print("Asymétrie (skew) :", round(c.skew(), 2))

**Interprétation.** La distribution est centrée autour de 26 kWh par jour (moyenne 26,2 ; médiane 25,9 ; écart-type 10,0) avec une queue à droite (asymétrie 0,56) : quelques journées dépassent 50 kWh, jusqu'à 79,6. On observe aussi un petit groupe de journées à très faible consommation (environ 5 à 9 kWh), séparé du corps principal, qui correspond probablement aux périodes d'absence de l'été.

In [ ]:
# Graphique 3 : profil par jour de la semaine et par mois
jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
par_jour = c.groupby(c.index.dayofweek).mean()
par_mois = c.groupby(c.index.month).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(jours, par_jour.values)
axes[0].set_title("Consommation moyenne par jour de la semaine")
axes[0].set_ylabel("Énergie moyenne (kWh par jour)")
axes[1].bar(par_mois.index.astype(str), par_mois.values)
axes[1].set_title("Consommation moyenne par mois")
axes[1].set_xlabel("Mois (1 = janvier)"); axes[1].set_ylabel("Énergie moyenne (kWh par jour)")
plt.tight_layout(); plt.show()

print("Par jour de semaine (kWh) :", dict(zip(jours, par_jour.round(2))))

**Interprétation.** Le week-end consomme plus que la semaine : 29,9 kWh le samedi et 29,3 kWh le dimanche contre 23,5 à 26,0 kWh les autres jours, soit environ 19 % de plus en moyenne. La saisonnalité est forte : décembre et janvier sont les mois les plus consommateurs (environ 35 kWh par jour) et août le plus bas (environ 13 kWh). La baseline « comme hier » devrait donc se tromper surtout le lundi (qui suit un dimanche élevé) et le samedi (qui suit un vendredi plus bas).

In [ ]:
# Graphique 4 : autocorrélation (lien de la série avec son propre passé)
lags = range(1, 31)
acf = [c.autocorr(lag=k) for k in lags]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(list(lags), acf)
ax.set_title("Autocorrélation de la consommation journalière")
ax.set_xlabel("Décalage k (jours)"); ax.set_ylabel("Corrélation entre y(t) et y(t−k)")
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

for k in (1, 2, 7, 14):
    print(f"k = {k:>2} jours : r = {c.autocorr(lag=k):.2f}")

**Interprétation.** L'autocorrélation est la plus forte à 1 jour (r = 0,61). On observe un léger pic à 7 jours (r = 0,55), plus haut que ses voisins, ce qui confirme un rythme hebdomadaire ; à 14 jours, r = 0,48. Elle reste supérieure à 0,2 jusqu'à 30 jours, signe de la saisonnalité annuelle : les jours d'une même période de l'année se ressemblent. Nous retenons donc comme variables `lag_1`, `lag_7` et une moyenne de la semaine précédente (`mm7`), complétées par le calendrier.

## 5 · Audit de fuite de données
Une variable est utilisable si et seulement si elle est calculable avec ce que l'on sait à l'instant de la prévision : ici, la fin du jour J−1 pour prévoir le jour J.

| Variable | Utilisable pour prédire `conso_kwh(J)` ? | Pourquoi |
|---|---|---|
| `Global_active_power`, `puissance_moy_kw` du jour J | ❌ Fuite | c'est la cible elle-même (`conso_kwh` = puissance moyenne × 24) |
| `Global_intensity` du jour J | ❌ Fuite | quasi dérivée de la puissance (P ≈ U × I) |
| `Voltage`, `Global_reactive_power` du jour J | ❌ Fuite | mesurés le jour J, donc inconnus à J−1 |
| `Sub_metering_1/2/3` et `reste_kwh` du jour J | ❌ Fuite | leur somme reconstitue la consommation du jour J |
| Les mêmes variables du jour J−1 ou avant (*lags*) | ✅ | connues à la fin de J−1 |
| Calendrier (jour de semaine, mois, week-end) | ✅ | connu longtemps à l'avance |
| Moyenne mobile **centrée** | ❌ | contient les jours après la date de prévision |
| Moyenne de **toute** la série (imputation, standardisation) | ❌ | contient le futur et le jeu de test |

Nous excluons donc toutes les mesures du jour à prévoir ; seuls les décalages d'au moins 1 jour et le calendrier sont utilisés. Le test automatique de la section 6 vérifie que falsifier le futur ne change aucune variable passée.

## 6 · Variables construites
Pour chaque jour *t* à prévoir, nous construisons des variables **causales**, calculées avec des données antérieures à *t* uniquement.

| Variable | Définition | Justification métier |
|---|---|---|
| `lag_1` | consommation de la veille (t−1) | niveau récent : forte autocorrélation à 1 jour |
| `lag_7` | consommation du même jour la semaine dernière (t−7) | rythme hebdomadaire (semaine / week-end) |
| `mm7` | moyenne de t−7 à t−1 (fenêtre **décalée** de 1) | niveau de fond de la dernière semaine |
| `est_weekend`, `jour_semaine`, `mois` | calendrier | connu à l'avance ; capte la saison et le week-end |

**Jours incomplets :** pour les décalages, un jour manquant est remplacé par la **dernière valeur connue** (`ffill`, limitée à 3 jours). Cette règle est causale : elle n'utilise que le passé. La **cible** reste `NaN` quand le jour est incomplet ; ces lignes sont exclues de l'évaluation. Le plus long bloc de jours manquants fait 6 jours, donc quelques jours auront un décalage vide et seront également écartés.

In [ ]:
def construire_features(conso):
    # conso : série journalière indexée par date (kWh). Retourne la table cible + variables.
    c_ff = conso.ffill(limit=3)                       # dernière valeur connue (causal)
    f = pd.DataFrame(index=conso.index)
    f["conso_kwh"] = conso                             # cible : consommation du jour t
    f["lag_1"] = c_ff.shift(1)                         # veille
    f["lag_7"] = c_ff.shift(7)                         # même jour, semaine précédente
    f["mm7"] = c_ff.shift(1).rolling(7, min_periods=5).mean()   # on décale AVANT de glisser
    f["est_weekend"] = (conso.index.dayofweek >= 5).astype(int)
    f["jour_semaine"] = conso.index.dayofweek
    f["mois"] = conso.index.month
    return f

FEATURES = ["lag_1", "lag_7", "mm7", "est_weekend", "jour_semaine", "mois"]
feat = construire_features(daily["conso_kwh"])
print("Lignes :", len(feat), "| Lignes avec cible ET toutes les variables :",
      int(feat[["conso_kwh"] + FEATURES].notna().all(axis=1).sum()))
feat.head(12).round(2)

### Test anti-fuite : falsifier le futur
On choisit une date T et on remplace tout ce qui vient **après T** par des valeurs absurdes (9 999 kWh), puis on recalcule les variables. Si le code est correct, les variables des jours ≤ T+1 ne changent pas (le jour T+1 est prévu avec ce que l'on sait à la fin de T). Si elles changent, il y a une fuite.

In [ ]:
T = pd.Timestamp("2009-06-30")
conso_originale = daily["conso_kwh"]

conso_falsifiee = conso_originale.copy()
conso_falsifiee[conso_falsifiee.index > T] = 9999.0           # on salit le futur

f_orig = construire_features(conso_originale)
f_fals = construire_features(conso_falsifiee)

jusqu_a = T + pd.Timedelta(days=1)                            # horizon h = 1 jour
a = f_orig.loc[:jusqu_a, FEATURES].to_numpy(dtype=float)
b = f_fals.loc[:jusqu_a, FEATURES].to_numpy(dtype=float)
assert np.allclose(a, b, equal_nan=True), "FUITE DÉTECTÉE : une variable utilise le futur !"
print("Test anti-fuite réussi : aucune variable des jours <= T+1 ne dépend du futur.")

# Contre-exemple : une moyenne mobile CENTRÉE utilise le futur
fuite = conso_originale.rolling(3, center=True).mean()
fuite_fals = conso_falsifiee.rolling(3, center=True).mean()
print("La moyenne centrée change-t-elle quand on salit le futur ?",
      not np.allclose(fuite.loc[:jusqu_a].to_numpy(dtype=float),
                      fuite_fals.loc[:jusqu_a].to_numpy(dtype=float), equal_nan=True))

**Interprétation.** Le test est réussi : aucune de nos variables passées ne dépend du futur. À l'inverse, une moyenne mobile centrée change quand on falsifie le futur (`True`), ce qui montre qu'elle fuit et justifie de l'écarter.

## 7 · Baselines de calibrage du critère de succès
Nous mesurons l'erreur des deux règles simples (« comme hier », « comme la semaine dernière ») pour calibrer le critère de succès. L'évaluation est faite sur **2009** et non sur le jeu de test : le test (période la plus récente) n'est utilisé qu'une seule fois, à la fin.

La MAE est la moyenne des |réel − prévu|, exprimée en kWh par jour.

In [ ]:
ref = feat.loc["2009-01-01":"2009-12-31"].dropna(subset=["conso_kwh", "lag_1", "lag_7"])

mae_naive = (ref["conso_kwh"] - ref["lag_1"]).abs().mean()      # « comme hier »
mae_saison = (ref["conso_kwh"] - ref["lag_7"]).abs().mean()     # « comme la semaine dernière »
meilleure = min(mae_naive, mae_saison)

print(f"Jours évalués (2009)          : {len(ref)}")
print(f"MAE naïve (comme hier)        : {mae_naive:.2f} kWh/jour")
print(f"MAE saisonnière (sem. passée) : {mae_saison:.2f} kWh/jour")
print(f"Meilleure baseline            : {meilleure:.2f} kWh/jour")
print(f"Cible à -15 %                 : {0.85 * meilleure:.2f} kWh/jour")

**Interprétation.** Sur les 361 jours évalués en 2009, la baseline « même jour la semaine dernière » (MAE 5,77 kWh/jour) fait mieux que « comme hier » (6,29), ce qui est cohérent avec le rythme hebdomadaire observé en section 4. L'écart est modeste (environ 8 %). L'erreur de la meilleure baseline représente environ 22 % de la consommation moyenne (5,77 sur 26,2).

Le critère de succès conserve la cible de −15 %, soit **4,90 kWh/jour**. Elle est ambitieuse mais plausible grâce au calendrier et à la saisonnalité. Elle est fixée avant toute évaluation sur le jeu de test et ne sera pas modifiée.

## 8 · Profils horaires de chaque journée (problématique 2)
**Objectif :** pour chaque jour, construire un vecteur de **24 valeurs** : la puissance moyenne de chaque heure (0 h à 23 h). Chaque jour devient un point dans un espace à 24 dimensions, exploité par PCA, K-Means et DBSCAN dans le notebook 02.

Le clustering regroupe des observations qui se **ressemblent** : deux journées ont le même profil si leurs courbes de 24 h se ressemblent, même si leurs totaux diffèrent un peu.

**Traitement :** nous gardons les jours complets (couverture ≥ 95 %) et retirons ceux qui ont une heure entièrement absente. Ces jours sont **supprimés** et non remplis : en clustering, il n'y a pas de notion de futur. Les 24 colonnes seront **standardisées** dans le notebook 02, avant la PCA et les algorithmes à base de distance.

In [ ]:
horaire = raw["Global_active_power"].resample("h").mean()                 # puissance moyenne par heure
tab = horaire.to_frame("p")
tab["date"] = tab.index.normalize()
tab["heure"] = tab.index.hour
profils = tab.pivot(index="date", columns="heure", values="p")            # 1 ligne = 1 jour, 24 colonnes

jours_ok = daily.index[~daily["jour_incomplet"]]
profils = profils.loc[profils.index.isin(jours_ok)]
avant = len(profils)
profils = profils.dropna()                                                # jours avec une heure absente
print(f"Jours complets : {avant} | retirés (heure absente) : {avant - len(profils)} | conservés : {len(profils)}")
profils.columns = [f"h{h:02d}" for h in profils.columns]
profils.head(3).round(2)

Aucun jour complet n'a été retiré : les 1 417 jours conservés ont donc leurs 24 heures renseignées.

In [ ]:
# Courbe moyenne et dispersion d'un jour type, pour vérifier que les profils ont du sens
moy = profils.mean()
q10, q90 = profils.quantile(0.10), profils.quantile(0.90)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(24), moy.values, marker="o", label="Moyenne de tous les jours")
ax.fill_between(range(24), q10.values, q90.values, alpha=0.25, label="Entre le 10e et le 90e centile")
ax.set_title("Profil horaire moyen d'une journée")
ax.set_xlabel("Heure de la journée"); ax.set_ylabel("Puissance moyenne (kW)")
ax.legend(); plt.tight_layout(); plt.show()

**Interprétation.** La courbe moyenne montre un creux nocturne (environ 0,4 à 0,6 kW entre 1 h et 5 h), une montée rapide vers 7 h (environ 1,4 kW), un plateau en journée (environ 1,1 à 1,3 kW), un léger creux en fin d'après-midi (environ 0,8 kW vers 17 h), puis un pic en soirée entre 20 h et 22 h (environ 1,6 kW) avant la redescente de la nuit. La bande entre le 10e et le 90e centile est large (jusqu'à environ 2,5 kW le matin et environ 3 kW le soir) : l'amplitude des pics varie fortement d'une journée à l'autre, ce qui laisse penser à plusieurs types de journées. Le clustering du notebook 02 permettra de le vérifier.

## 9 · Sauvegarde et bilan
Les tables propres sont enregistrées pour les notebooks suivants (quelques centaines de Ko).

In [ ]:
feat.to_csv(os.path.join(DATA_DIR, "features_journalieres.csv"), index_label="date")
daily.to_csv(os.path.join(DATA_DIR, "table_journaliere.csv"), index_label="date")
profils.to_csv(os.path.join(DATA_DIR, "profils_horaires.csv"), index_label="date")
print("Fichiers enregistrés dans", DATA_DIR + "/ :")
print(sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))

### Bilan de l'analyse des données
- **Qualité :** 1,25 % de valeurs manquantes, regroupées en 71 blocs (le plus long : 5 jours), sans doublon. 25 jours sur 1 442 sont incomplets ; 1 417 jours sont exploitables.
- **Cible :** consommation moyenne de 26,2 kWh/jour (écart-type 10,0), saisonnalité annuelle forte (hiver élevé, été bas), week-end environ 19 % plus élevé que la semaine, autocorrélation de 0,61 à 1 jour et 0,55 à 7 jours.
- **Fuites écartées :** toutes les mesures du jour à prévoir, les moyennes centrées et toute statistique calculée sur la série entière. Test « falsifier le futur » réussi.
- **Variables retenues :** `lag_1`, `lag_7`, `mm7`, `est_weekend`, `jour_semaine`, `mois`.
- **Baselines (2009) :** MAE de 6,29 kWh/jour (hier) et 5,77 kWh/jour (semaine dernière) ; objectif de 4,90 kWh/jour.
- **Limites :** un seul foyer, données de 2006 à 2010 (pas de météo, pas d'information sur les périodes d'absence), valeurs extrêmes en début de série.

### Suite
Le notebook 02 réalise la modélisation : split chronologique, validation croisée temporelle, régression linéaire et forêt aléatoire pour la prévision ; PCA, K-Means et DBSCAN sur les profils horaires.